# The Cross-Entropy Method: Solving RL Without Gradients

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/cross_entropy_method.ipynb)

This notebook implements the Cross-Entropy Method (CEM) for policy search on CartPole-v1.
CEM is a derivative-free optimisation algorithm: no backpropagation, no value functions,
just sample, evaluate, select the best, and refit.

**What you'll learn:**
- How CEM maintains and narrows a Gaussian distribution over policy parameters
- The "noisy" variant from Szita & Lörincz (2006) that prevents premature convergence
- How CEM compares to gradient-based methods like REINFORCE and DQN

**References:**
- Rubinstein (1999), "The Cross-Entropy Method for Combinatorial and Continuous Optimization"
- Szita & Lörincz (2006), "Learning Tetris Using the Noisy Cross-Entropy Method"

**Blog post:** [sesen.ai/blog/cross-entropy-method-evolution-style-rl](https://sesen.ai/blog/cross-entropy-method-evolution-style-rl)

In [ ]:
# Install dependencies (Colab)
# !pip install gymnasium matplotlib numpy

In [ ]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image as IPImage, display

np.random.seed(42)

## 1. The Quick Win: CEM Solves CartPole

Our policy is a linear function: `action = 1 if theta @ observation > 0 else 0`.
With just 4 parameters (one per observation dimension), CEM finds a perfect policy.

In [ ]:
def evaluate_policy(env_name, theta, n_episodes=1):
    """Evaluate a linear policy on the environment."""
    env = gym.make(env_name)
    total_reward = 0
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        ep_reward = 0
        while not done:
            action = 1 if np.dot(theta, obs) > 0 else 0
            obs, reward, terminated, truncated, _ = env.step(action)
            ep_reward += reward
            done = terminated or truncated
        total_reward += ep_reward
    env.close()
    return total_reward / n_episodes

In [ ]:
def cem(env_name, n_params, batch_size=200, n_iter=50, elite_frac=0.2,
        initial_std=1.0, extra_std=0.5, std_decay_time=25):
    """
    Cross-Entropy Method for policy search.

    Args:
        env_name: Gymnasium environment name
        n_params: Number of policy parameters
        batch_size: Number of parameter samples per iteration
        n_iter: Number of iterations
        elite_frac: Fraction of top-performing samples to keep
        initial_std: Initial standard deviation of parameter distribution
        extra_std: Extra noise for noisy CEM (Szita & Lörincz 2006)
        std_decay_time: Iterations over which extra noise decays to zero

    Returns:
        best_theta: Best policy parameters found
        history: Dict with reward history per iteration
    """
    n_elite = int(np.round(batch_size * elite_frac))
    th_mean = np.zeros(n_params)
    th_std = np.ones(n_params) * initial_std

    history = {
        'mean_rewards': [], 'max_rewards': [],
        'elite_mean_rewards': [], 'all_rewards': [],
        'th_means': [th_mean.copy()], 'th_stds': [th_std.copy()],
    }
    best_reward, best_theta = -np.inf, th_mean.copy()

    for iteration in range(n_iter):
        # Decaying extra noise (Szita & Lörincz 2006)
        noise_multiplier = max(1.0 - iteration / float(std_decay_time), 0)
        sample_std = np.sqrt(th_std + np.square(extra_std) * noise_multiplier)

        # Sample parameter vectors from Gaussian
        thetas = th_mean + sample_std * np.random.randn(batch_size, n_params)

        # Evaluate each candidate policy
        rewards = np.array([evaluate_policy(env_name, th) for th in thetas])

        # Select elite samples and refit distribution
        elite_inds = rewards.argsort()[-n_elite:]
        elite_thetas = thetas[elite_inds]
        th_mean = elite_thetas.mean(axis=0)
        th_std = elite_thetas.var(axis=0)

        if rewards.max() > best_reward:
            best_reward = rewards.max()
            best_theta = thetas[rewards.argmax()].copy()

        history['mean_rewards'].append(rewards.mean())
        history['max_rewards'].append(rewards.max())
        history['elite_mean_rewards'].append(rewards[elite_inds].mean())
        history['all_rewards'].append(rewards.copy())
        history['th_means'].append(th_mean.copy())
        history['th_stds'].append(th_std.copy())

        if (iteration + 1) % 10 == 0 or iteration == 0:
            print(f"Iter {iteration+1:3d} | Mean: {rewards.mean():6.1f} | "
                  f"Max: {rewards.max():6.0f} | Elite mean: {rewards[elite_inds].mean():6.1f}")

    return best_theta, history

In [ ]:
# Run CEM on CartPole-v1
env = gym.make('CartPole-v1')
n_params = env.observation_space.shape[0]  # 4
env.close()

best_theta, history = cem(
    'CartPole-v1',
    n_params=n_params,
    batch_size=200,
    n_iter=50,
    elite_frac=0.2,
    initial_std=1.0,
    extra_std=0.5,
    std_decay_time=25,
)

print(f"\nBest theta: {best_theta}")
print(f"Best reward: {max(history['max_rewards'])}")

In [ ]:
# Final evaluation over 100 episodes
final_scores = [evaluate_policy('CartPole-v1', best_theta) for _ in range(100)]
print(f"Final evaluation (100 episodes): {np.mean(final_scores):.0f} ± {np.std(final_scores):.0f}")

## 2. Visualising the Training

### Training Curve

The population mean reward climbs from ~67 to ~500 over 50 iterations.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
iters = range(1, len(history['mean_rewards']) + 1)
ax.plot(iters, history['mean_rewards'], 'b-', alpha=0.7, label='Population mean')
ax.plot(iters, history['elite_mean_rewards'], 'r-', linewidth=2, label='Elite mean')
ax.plot(iters, history['max_rewards'], 'g--', alpha=0.5, label='Best in batch')
ax.axhline(y=500, color='k', linestyle=':', alpha=0.3, label='Max possible (500)')
ax.set_xlabel('Iteration')
ax.set_ylabel('Total Reward')
ax.set_title('CEM Training on CartPole-v1')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

### Reward Distribution Over Time

The entire population shifts from low to high reward.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
idxs = [0, min(9, len(history['all_rewards'])-1), len(history['all_rewards'])-1]
titles = ['Iteration 1', f'Iteration {idxs[1]+1}', f'Iteration {idxs[2]+1}']

for ax, idx, title in zip(axes, idxs, titles):
    rewards = history['all_rewards'][idx]
    ax.hist(rewards, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(x=rewards.mean(), color='red', linestyle='--', label=f'Mean: {rewards.mean():.0f}')
    ax.set_xlabel('Episode Reward')
    ax.set_ylabel('Count')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_xlim(0, 520)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Convergence Animation

Watch the reward distribution collapse onto the solution over 50 iterations.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

def update(frame):
    ax.clear()
    rewards = history['all_rewards'][frame]
    n_elite = int(np.round(200 * 0.2))
    elite_threshold = np.sort(rewards)[-n_elite]

    ax.hist(rewards, bins=20, color='steelblue', edgecolor='white', alpha=0.8, range=(0, 520))
    ax.axvline(x=rewards.mean(), color='red', linestyle='--', linewidth=2,
               label=f'Mean: {rewards.mean():.0f}')
    ax.axvline(x=elite_threshold, color='orange', linestyle=':', linewidth=2,
               label=f'Elite cutoff: {elite_threshold:.0f}')
    ax.set_xlim(0, 520)
    ax.set_ylim(0, 120)
    ax.set_xlabel('Episode Reward')
    ax.set_ylabel('Count')
    ax.set_title(f'CEM Iteration {frame + 1}')
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)

n_frames = min(12, len(history['all_rewards']))
frame_indices = np.linspace(0, len(history['all_rewards'])-1, n_frames, dtype=int)
anim = FuncAnimation(fig, update, frames=frame_indices, interval=500)
anim.save('/tmp/cem_convergence.gif', writer=PillowWriter(fps=3), dpi=100)
plt.close(fig)

display(IPImage(filename='/tmp/cem_convergence.gif'))

## 3. CEM vs Random Search

Both methods sample 200 policies per iteration. The difference: CEM builds on what
worked, while random search starts fresh every time.

In [ ]:
# Random search baseline: 50 iterations x 200 random samples
np.random.seed(42)
random_mean_rewards = []
for i in range(50):
    random_thetas = np.random.randn(200, n_params)
    random_rewards = [evaluate_policy('CartPole-v1', th) for th in random_thetas]
    random_mean_rewards.append(np.mean(random_rewards))
    if (i + 1) % 10 == 0:
        print(f"Random search iter {i+1}: mean = {np.mean(random_rewards):.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 51), history['mean_rewards'], 'b-', linewidth=2, label='CEM (population mean)')
ax.plot(range(1, 51), random_mean_rewards, 'r--', linewidth=2, label='Random search (mean of 200)')
ax.axhline(y=500, color='k', linestyle=':', alpha=0.3, label='Max possible (500)')
ax.set_xlabel('Iteration')
ax.set_ylabel('Mean Reward (200 samples)')
ax.set_title('CEM vs Random Search: Population Mean Reward')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 4. Understanding the Algorithm

### The Three Steps

Each CEM iteration:

1. **Sample** — Draw `batch_size` parameter vectors from $\mathcal{N}(\mu, \sigma^2)$
2. **Evaluate and Select** — Run each as a policy, keep the top `elite_frac`
3. **Refit** — Set $\mu$ and $\sigma^2$ to the mean and variance of the elite set

### The Noisy Variant

The original CEM can collapse its variance to zero too quickly. Szita & Lörincz (2006)
added decaying extra variance:

$$\sigma_{t+1}^2 = \sigma_{t,\text{elite}}^2 + Z_t^2 \cdot \sigma_{\text{extra}}^2$$

where $Z_t = \max(1 - t/T_{\text{decay}}, 0)$ decays linearly.

## 5. Exercises

### Exercise 1: Elite Fraction Sweep

Try `elite_frac` values of 0.01, 0.1, 0.2, and 0.5. How does selectivity affect
convergence speed and stability?

In [ ]:
# Your code here
elite_fracs = [0.01, 0.1, 0.2, 0.5]

fig, ax = plt.subplots(figsize=(8, 5))
for ef in elite_fracs:
    np.random.seed(42)
    _, hist = cem('CartPole-v1', n_params=4, batch_size=200, n_iter=30,
                  elite_frac=ef, initial_std=1.0, extra_std=0.5, std_decay_time=15)
    ax.plot(range(1, 31), hist['mean_rewards'], label=f'elite_frac={ef}')

ax.set_xlabel('Iteration')
ax.set_ylabel('Population Mean Reward')
ax.set_title('Effect of Elite Fraction on Convergence')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

### Exercise 2: Noisy vs Vanilla CEM

Set `extra_std=0` (vanilla CEM) and compare with `extra_std=0.5` (noisy CEM).
Does the noise help on CartPole?

In [ ]:
# Your code here

### Exercise 3: Neural Network Policy

Replace the linear policy `theta @ obs` with a small neural network (e.g., 8 hidden units
with tanh activation). CEM now optimises the flattened weight vector.
How many iterations does it take? At what network size does CEM become impractical?

In [ ]:
# Your code here
# Hint: n_params = 4*8 + 8 + 8*2 + 2 = 58 for a (4, 8, 2) network

### Exercise 4: Different Environments

Try CEM on `Acrobot-v1` or `MountainCar-v0`. Which environments does CEM handle well,
and which expose its limitations?

In [ ]:
# Your code here